In [1]:
import random
import copy

GRID = 6
start = (0,0)
goal = (5,5)

print("=== Adaptive Execution with Monitoring ===\n")

# ------------------------------------------------
# Generate obstacles
# ------------------------------------------------

obstacles=[]

while len(obstacles)<4:
    r=random.randint(0,GRID-1)
    c=random.randint(0,GRID-1)

    if (r,c)!=start and (r,c)!=goal:
        obstacles.append((r,c))

# ------------------------------------------------
# Display grid state
# ------------------------------------------------

def show_grid(drone_pos):

    grid=[["." for _ in range(GRID)] for _ in range(GRID)]

    grid[start[0]][start[1]]="S"
    grid[goal[0]][goal[1]]="G"

    for o in obstacles:
        grid[o[0]][o[1]]="X"

    r,c=drone_pos
    grid[r][c]="D"

    for row in grid:
        print(row)

# ------------------------------------------------
# Initial plan
# ------------------------------------------------

def generate_plan(current):

    r,c=current
    path=[current]

    while (r,c)!=goal:

        if c<goal[1]:
            c+=1
        elif r<goal[0]:
            r+=1

        path.append((r,c))

    return path

# ------------------------------------------------
# Replanning logic
# ------------------------------------------------

def replan(current):

    r,c=current
    path=[current]

    while (r,c)!=goal:

        if c<goal[1] and (r,c+1) not in obstacles:
            c+=1

        elif r<goal[0] and (r+1,c) not in obstacles:
            r+=1

        else:
            r=min(GRID-1,r+1)

        path.append((r,c))

    return path


# ------------------------------------------------
# Start execution
# ------------------------------------------------

position=start
plan=generate_plan(position)

print("Initial Environment\n")
show_grid(position)
print("Planned Path:",plan)
step_number=1

while position!=goal:

    print("\n============================")
    print("STEP",step_number)
    print("============================")

    next_step=plan[1]

    print("\nDrone attempting move →",next_step)

    # --------------------------
    # ACTION MONITORING
    # --------------------------

    action_success=random.choice([True,True,True,False])

    if not action_success:

        print("\n⚠ ACTION MONITORING")
        print("Motor execution failed")

        plan=replan(position)

        print("Replanned Path:",plan)
        continue


    # --------------------------
    # STATE MONITORING
    # --------------------------

    if next_step in obstacles:

        print("\n⚠ STATE MONITORING")
        print("Unexpected obstacle detected at",next_step)

        plan=replan(position)

        print("Replanned Path:",plan)
        continue


    # --------------------------
    # PLAN MONITORING
    # --------------------------

    future_path=plan[2:]

    for future in future_path:

        if future in obstacles:

            print("\n⚠ PLAN MONITORING")
            print("Future path predicted blocked at",future)

            plan=replan(position)

            print("Replanned Path:",plan)
            break

    # move drone
    position=next_step
    plan=plan[1:]

    print("\nDrone moved to",position)

    print("\nCurrent Grid State\n")
    show_grid(position)

    step_number+=1


print("\n🎯 GOAL REACHED")

=== Adaptive Execution with Monitoring ===

Initial Environment

['D', '.', '.', '.', 'X', '.']
['.', '.', '.', '.', 'X', '.']
['.', '.', '.', '.', '.', '.']
['.', 'X', '.', '.', '.', '.']
['.', 'X', '.', '.', '.', '.']
['.', '.', '.', '.', '.', 'G']
Planned Path: [(0, 0), (0, 1), (0, 2), (0, 3), (0, 4), (0, 5), (1, 5), (2, 5), (3, 5), (4, 5), (5, 5)]

STEP 1

Drone attempting move → (0, 1)

⚠ PLAN MONITORING
Future path predicted blocked at (0, 4)
Replanned Path: [(0, 0), (0, 1), (0, 2), (0, 3), (1, 3), (2, 3), (2, 4), (2, 5), (3, 5), (4, 5), (5, 5)]

Drone moved to (0, 1)

Current Grid State

['S', 'D', '.', '.', 'X', '.']
['.', '.', '.', '.', 'X', '.']
['.', '.', '.', '.', '.', '.']
['.', 'X', '.', '.', '.', '.']
['.', 'X', '.', '.', '.', '.']
['.', '.', '.', '.', '.', 'G']

STEP 2

Drone attempting move → (0, 2)

⚠ ACTION MONITORING
Motor execution failed
Replanned Path: [(0, 1), (0, 2), (0, 3), (1, 3), (2, 3), (2, 4), (2, 5), (3, 5), (4, 5), (5, 5)]

STEP 2

Drone attempting move 